### **IMPORTO LIBRERIE**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### **IMPORTO DATI**

In [ ]:
#Importo i dataframe precedentemente puliti per l'analisi
df_software = pd.read_csv("datas/processed/dataset_software.csv")
df_hardware = pd.read_csv("datas/processed/dataset_hardware.csv")
df_software_test = pd.read_csv("datas/processed/dataset_software_test.csv")
df_hardware_test = pd.read_csv("datas/processed/dataset_hardware_test.csv")

### **TABELLE**

In [ ]:
colonne_attivo = [
    "Totale Attivo migl USD 2024",
    "Totale Attivo migl USD 2023",
    "Totale Attivo migl USD 2022",
    "Totale Attivo migl USD 2021",
    "Totale Attivo migl USD 2020",
]

colonne_descrittive = [
    "roe_medio",
    "roa_medio",
    "debt_equity_medio",
    "liquidita_media",
    "attivo_medio",
    "log_eta_azienda",
]

etichette_variabili = {
    "roe_medio": "ROE medio",
    "roa_medio": "ROA medio",
    "debt_equity_medio": "Debt/Equity medio",
    "liquidita_media": "Indice di liquidità medio",
    "attivo_medio": "Totale attivo medio (migliaia di USD)",
    "log_eta_azienda": "Età azienda (ln(1+età))",
}

etichette_statistiche = {
    "mean": "Media",
    "median": "Mediana",
    "std": "Deviazione standard",
    "min": "Minimo",
    "max": "Massimo",
}

cartella_export_descrittiva = Path("statistica_descrittiva_export/tex")


def aggiungi_attivo_medio(df):
    df = df.copy()
    df["attivo_medio"] = df[colonne_attivo].astype(float).mean(axis=1, skipna=True)
    return df


def crea_tabella_statistica_descrittiva(df, titolo, nome_file):
    tabella = df[colonne_descrittive].agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ]).T

    tabella = tabella.rename(
        index=etichette_variabili,
        columns=etichette_statistiche
    )
    tabella.index.name = "Indicatore"

    cartella_export_descrittiva.mkdir(exist_ok=True)
    percorso_tex = cartella_export_descrittiva / nome_file
    tabella_export = tabella.reset_index()
    tabella_export.to_latex(
        percorso_tex,
        caption=titolo,
        label=f"tab:{percorso_tex.stem}",
        escape=True,
        index=False,
        float_format="{:,.2f}".format,
        column_format="lrrrrr",
    )

    return tabella


def formatta_tabella_descrittiva(tabella, titolo):
    return (
        tabella.style
        .set_caption(titolo)
        .format("{:,.2f}")
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("font-size", "1.1em")]},
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "right")]},
        ])
    )


df_software = aggiungi_attivo_medio(df_software)
df_hardware = aggiungi_attivo_medio(df_hardware)
df_completo = pd.concat([df_software, df_hardware], ignore_index=True)

statistica_descrittiva_totale = crea_tabella_statistica_descrittiva(
    df_completo,
    "Statistiche descrittive - Campione totale",
    "statistica_descrittiva_totale.tex"
)

statistica_descrittiva_hardware = crea_tabella_statistica_descrittiva(
    df_hardware,
    "Statistiche descrittive - Hardware",
    "statistica_descrittiva_hardware.tex"
)

statistica_descrittiva_software = crea_tabella_statistica_descrittiva(
    df_software,
    "Statistiche descrittive - Software",
    "statistica_descrittiva_software.tex"
)

stile_statistica_descrittiva_totale = formatta_tabella_descrittiva(
    statistica_descrittiva_totale,
    "Statistiche descrittive - Campione totale"
)
stile_statistica_descrittiva_hardware = formatta_tabella_descrittiva(
    statistica_descrittiva_hardware,
    "Statistiche descrittive - Hardware"
)
stile_statistica_descrittiva_software = formatta_tabella_descrittiva(
    statistica_descrittiva_software,
    "Statistiche descrittive - Software"
)

percorsi_export_descrittiva = sorted(cartella_export_descrittiva.glob("*.tex"))
percorsi_export_descrittiva

In [ ]:
stile_statistica_descrittiva_totale

In [ ]:
stile_statistica_descrittiva_hardware

In [ ]:
stile_statistica_descrittiva_software

### **DISTRIBUZIONI**

Dato che il ROE medio e mediano è negativo, penso sia interessante analizzare il modo in cui è distribuito (prevedendo ovviamente un'asimmetria a sinistra). Replico lo stesso tipo di istogramma anche per ROA, D/E e CR (liquidity), esportando i grafici in formato PNG.

In [ ]:
from pathlib import Path


def mostra_istogramma_variabile(df, nome_database, colonna, etichetta, nome_variabile, percorso_export=None, bins=30):
    import matplotlib.pyplot as plt

    valori = df[colonna].dropna()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(valori, bins=bins, color="#4C78A8", edgecolor="black", alpha=0.85)
    ax.axvline(valori.mean(), color="#E45756", linestyle="--", linewidth=2, label=f"Media: {valori.mean():.2f}")
    ax.axvline(valori.median(), color="#54A24B", linestyle="-", linewidth=2, label=f"Mediana: {valori.median():.2f}")
    ax.set_title(f"Distribuzione del {nome_variabile} - {nome_database}")
    ax.set_xlabel(etichetta)
    ax.set_ylabel("Numero di aziende")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()

    if percorso_export is not None:
        fig.savefig(percorso_export, dpi=300, bbox_inches="tight")

    plt.show()

    return fig, ax


cartella_export_grafici = Path("statistica_descrittiva_export/grafici")
cartella_export_grafici.mkdir(parents=True, exist_ok=True)

variabili_istogrammi = [
    ("roe_medio", "ROE", "roe"),
    ("roa_medio", "ROA", "roa"),
    ("debt_equity_medio", "D/E", "debt_equity"),
    ("liquidita_media", "CR (liquidity)", "cr_liquidity"),
]

for colonna, nome_variabile, nome_file in variabili_istogrammi:
    etichetta = etichette_variabili[colonna]

    mostra_istogramma_variabile(
        df_software,
        "Software",
        colonna,
        etichetta,
        nome_variabile,
        cartella_export_grafici / f"istogramma_{nome_file}_software.png",
    );

    mostra_istogramma_variabile(
        df_hardware,
        "Hardware",
        colonna,
        etichetta,
        nome_variabile,
        cartella_export_grafici / f"istogramma_{nome_file}_hardware.png",
    );

### **CONFRONTO SOFTWARE E HARDWARE**

In [ ]:
variabili_confronto_sw_hw = {
    "roe_medio": "ROE",
    "roa_medio": "ROA",
    "debt_equity_medio": "Debt/Equity",
    "liquidita_media": "Indice di liquidità",
    "log_attivo_medio": "log(Totale Attivo)",
    "eta_azienda": "Età (anni)",
}

colonne_confronto_sw_hw = ["N", "Media", "Mediana", "Dev. Std.", "Min", "Max"]


def riepiloga_gruppo(df, colonna):
    serie = pd.to_numeric(df[colonna], errors="coerce")
    return pd.Series(
        {
            "N": int(serie.count()),
            "Media": serie.mean(),
            "Mediana": serie.median(),
            "Dev. Std.": serie.std(),
            "Min": serie.min(),
            "Max": serie.max(),
        },
        index=colonne_confronto_sw_hw,
    )


def crea_tabella_confronto_sw_hw(df_software, df_hardware):
    righe = []
    indici = []

    for colonna, etichetta in variabili_confronto_sw_hw.items():
        statistiche_software = riepiloga_gruppo(df_software, colonna)
        statistiche_hardware = riepiloga_gruppo(df_hardware, colonna)
        differenza = pd.Series(pd.NA, index=colonne_confronto_sw_hw)
        differenza["Media"] = statistiche_software["Media"] - statistiche_hardware["Media"]
        differenza["Mediana"] = statistiche_software["Mediana"] - statistiche_hardware["Mediana"]

        for gruppo, statistiche in [
            ("Software", statistiche_software),
            ("Hardware", statistiche_hardware),
            ("Differenza (Sw-Hw)", differenza),
        ]:
            righe.append(statistiche)
            indici.append((etichetta, gruppo))

    tabella = pd.DataFrame(
        righe,
        index=pd.MultiIndex.from_tuples(indici, names=["Indicatore", "Gruppo"]),
    )
    return tabella[colonne_confronto_sw_hw]


def formatta_tabella_confronto_sw_hw(tabella, titolo):
    tabella_formattata = tabella.copy().astype("object")

    for indice in tabella.index:
        riga_differenza = indice[1] == "Differenza (Sw-Hw)"
        for colonna in tabella.columns:
            valore = tabella.loc[indice, colonna]
            if pd.isna(valore):
                valore_formattato = ""
            elif colonna == "N":
                valore_formattato = f"{int(valore)}"
            elif riga_differenza and colonna in ["Media", "Mediana"]:
                valore_formattato = f"{valore:+,.2f}"
            else:
                valore_formattato = f"{valore:,.2f}"

            tabella_formattata.loc[indice, colonna] = valore_formattato

    return (
        tabella_formattata.style
        .set_caption(titolo)
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold"), ("font-size", "1.1em")]},
            {"selector": "th", "props": [("text-align", "center")]},
            {"selector": "td", "props": [("text-align", "right")]},
        ])
    )


statistica_descrittiva_confronto_sw_hw = crea_tabella_confronto_sw_hw(df_software, df_hardware)

tabella_confronto_sw_hw_export = formatta_tabella_confronto_sw_hw(
    statistica_descrittiva_confronto_sw_hw,
    "Statistiche descrittive - Confronto Software e Hardware",
).data.reset_index()

cartella_export_descrittiva.mkdir(parents=True, exist_ok=True)
tabella_confronto_sw_hw_export.to_latex(
    cartella_export_descrittiva / "statistica_descrittiva_confronto_sw_hw.tex",
    caption="Statistiche descrittive - Confronto Software e Hardware",
    label="tab:statistica_descrittiva_confronto_sw_hw",
    escape=True,
    index=False,
    column_format="llrrrrrr",
)

stile_statistica_descrittiva_confronto_sw_hw = formatta_tabella_confronto_sw_hw(
    statistica_descrittiva_confronto_sw_hw,
    "Statistiche descrittive - Confronto Software e Hardware",
)

stile_statistica_descrittiva_confronto_sw_hw

### **MATRICE DI CORRELAZIONE**

In [ ]:



variabili_correlazione = {
    "roe_medio": "ROE",
    "roa_medio": "ROA",
    "debt_equity_medio": "D/E",
    "liquidita_media": "CR",
    "log_attivo_medio": "log_Attivo",
    "log_eta_azienda": "log_età",
}

dati_correlazione = df_completo[list(variabili_correlazione)].rename(columns=variabili_correlazione)
matrice_correlazione_pearson = dati_correlazione.corr(method="pearson")

cartella_export_grafici = Path("statistica_descrittiva_export/grafici")
cartella_export_grafici.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrice_correlazione_pearson,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlazione di Pearson"},
    ax=ax,
)
ax.set_title("Matrice di correlazione di Pearson - Campione totale")
ax.set_xlabel("")
ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
plt.setp(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
fig.savefig(cartella_export_grafici / "matrice_correlazione_pearson.png", dpi=300, bbox_inches="tight")
plt.show()